# VASONet

Set the training NPZ and output checkpoint. Select a single element type for the existing VASONet checkpoint architecture.

In [1]:
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vasonet import VASONet, quadrilateral

DATASET_PATH = Path("data/smoothing_data/quadrilateral.npz")
MODEL_PATH = Path("dualMLP_quad.pt")
SEED = 42
epochs = 300
patience = 20
batch_size = 64

torch.manual_seed(SEED)
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

In [2]:
def to_tensor(array):
    return torch.as_tensor(array, dtype=torch.float32)

with np.load(DATASET_PATH) as data:
    if "element_type" in data:
        raise ValueError("This VASONet trains on one element type at a time. Select a single-type NPZ.")
    coordinates, valence, targets = (
        to_tensor(data[key]) for key in ("coordinates", "valence", "target")
    )
    group_id = data["group_id"]

# Split original patches once; keep their cyclic reorderings together.
patches = np.random.default_rng(SEED).permutation(np.unique(group_id))

n_train, n_val = 7 * len(patches) // 10, 2 * len(patches) // 10
train_mask = torch.from_numpy(np.isin(group_id, patches[:n_train]))
val_mask = torch.from_numpy(np.isin(group_id, patches[n_train:n_train + n_val]))
test_mask = torch.from_numpy(np.isin(group_id, patches[n_train + n_val:]))

train_dataset = TensorDataset(coordinates[train_mask], valence[train_mask], targets[train_mask])
val_dataset = TensorDataset(coordinates[val_mask], valence[val_mask], targets[val_mask])
test_dataset = TensorDataset(coordinates[test_mask], valence[test_mask], targets[test_mask])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [3]:
from time import perf_counter

model = quadrilateral(coordinates.shape[1]).to(device)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=6.5e-4, betas=(0.9, 0.999))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.916)


def epoch_mse(loader, train=False):
    model.train(train)
    total = 0.0
    with torch.set_grad_enabled(train):
        for x, n, target in loader:
            x, n, target = x.to(device), n.to(device), target.to(device)
            error = loss(model(x, n), target)
            if train:
                optimizer.zero_grad(set_to_none=True)
                error.backward()
                optimizer.step()
            total += error.item() * len(target)
    return total / len(loader.dataset)


best_val = float("inf")
epochs_without_improvement = 0
history = {"train": [], "val": []}
started = perf_counter()
for epoch in range(1, epochs + 1):
    train_mse = epoch_mse(train_loader, train=True)
    val_mse = epoch_mse(val_loader)
    history["train"].append(train_mse)
    history["val"].append(val_mse)
    print(f"{epoch:03d}/{epochs} | Train {train_mse:.8f} | Val {val_mse:.8f}", flush=True)
    if val_mse < best_val:
        best_val = val_mse
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch}")
            break
    scheduler.step()

training_time = perf_counter() - started
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
best_train_mse = epoch_mse(train_loader)
test_mse = epoch_mse(test_loader)
print(f"Best epoch {best_epoch} | Val {best_val:.8f} | Test {test_mse:.8f}")
print(f"Saved best FP32 weights to {MODEL_PATH}")

001/300 | Train 0.00284368 | Val 0.00227885
002/300 | Train 0.00201088 | Val 0.00168810
003/300 | Train 0.00171784 | Val 0.00160962
004/300 | Train 0.00155446 | Val 0.00139702
005/300 | Train 0.00144349 | Val 0.00139893
006/300 | Train 0.00135247 | Val 0.00131901
007/300 | Train 0.00128284 | Val 0.00119242
008/300 | Train 0.00121946 | Val 0.00118041
009/300 | Train 0.00116555 | Val 0.00113209
010/300 | Train 0.00111951 | Val 0.00112261
011/300 | Train 0.00108240 | Val 0.00116007
012/300 | Train 0.00104218 | Val 0.00107475
013/300 | Train 0.00100984 | Val 0.00104411
014/300 | Train 0.00098120 | Val 0.00105171
015/300 | Train 0.00095326 | Val 0.00101319
016/300 | Train 0.00092876 | Val 0.00101992
017/300 | Train 0.00090714 | Val 0.00099333
018/300 | Train 0.00088321 | Val 0.00099074
019/300 | Train 0.00086634 | Val 0.00098320
020/300 | Train 0.00085032 | Val 0.00097996
021/300 | Train 0.00083272 | Val 0.00097489
022/300 | Train 0.00081716 | Val 0.00096296
023/300 | Train 0.00080484 | Val

In [7]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

total_epochs = len(history["train"])
epoch_axis = np.arange(1, total_epochs + 1)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epoch_axis, history["train"], label="Train", color="#0072B2")
ax.plot(epoch_axis, history["val"], label="Validation", color="#D55E00")
ax.axvline(best_epoch, color="0.5", linestyle="--", label=f"Best epoch: {best_epoch}")
ax.set(xlabel="Epoch", ylabel="MSE")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

# MSEs use the best checkpoint in eval mode; timing covers the training loop.
rows = [
    "| Best epoch | Total epochs | Train MSE | Val MSE | Test MSE | Training time (s) |",
    "|---:|---:|---:|---:|---:|---:|",
    f"| {best_epoch} | {total_epochs} | {best_train_mse:.8f} | {best_val:.8f} | {test_mse:.8f} | {training_time:.2f} |",
]
display(Markdown("\n".join(rows)))

NameError: name 'history' is not defined